# 04 — Inference Pipeline

End-to-end prediction pipeline supporting **two model backends**:

| Backend | Features | Gender Model | Race Model |
|---|---|---|---|
| **RF (baseline)** | 26 letter counts | `gender_prediction_model_rf.pkl` | `race_prediction_model_rf.pkl` |
| **LightGBM (improved)** | TF-IDF char n-grams | `gender_lgbm_model.pkl` | `race_lgbm_model.pkl` |

This notebook loads both, demonstrates single and batch prediction, and compares outputs side by side.

In [ ]:
import os
import gzip
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE       = os.path.abspath(os.path.join(os.getcwd(), '..'))
MODELS_DIR = os.path.join(BASE, 'models')
print('Base:', BASE)

## 1. Load All Models

In [ ]:
# ── RF baseline ────────────────────────────────────────────────────────────────
rf_gender = joblib.load(os.path.join(MODELS_DIR, 'gender_prediction_model_rf.pkl'))
rf_race   = joblib.load(os.path.join(MODELS_DIR, 'race_prediction_model_rf.pkl'))
print('RF gender  :', type(rf_gender).__name__)
print('RF race    :', type(rf_race).__name__)

# ── SVM alternative ───────────────────────────────────────────────────────────
svm_gz_path = os.path.join(MODELS_DIR, 'svm_gender_model_Nov_2024.pkl.gz')
with gzip.open(svm_gz_path, 'rb') as f:
    svm_gender = joblib.load(f)
print('SVM gender :', type(svm_gender).__name__)

In [ ]:
# ── LightGBM + TF-IDF (improved) ──────────────────────────────────────────────
lgbm_gender_vec = joblib.load(os.path.join(MODELS_DIR, 'gender_tfidf_vectorizer.pkl'))
lgbm_gender     = joblib.load(os.path.join(MODELS_DIR, 'gender_lgbm_model.pkl'))
lgbm_race_vec   = joblib.load(os.path.join(MODELS_DIR, 'race_tfidf_vectorizer.pkl'))
lgbm_race       = joblib.load(os.path.join(MODELS_DIR, 'race_lgbm_model.pkl'))

print('LGBM gender:', type(lgbm_gender).__name__)
print('LGBM race  :', type(lgbm_race).__name__)
print(f'  Gender TF-IDF features: {len(lgbm_gender_vec.get_feature_names_out()):,}')
print(f'  Race   TF-IDF features: {len(lgbm_race_vec.get_feature_names_out()):,}')

In [ ]:
# ── Shared label encoder ──────────────────────────────────────────────────────
race_le = joblib.load(os.path.join(MODELS_DIR, 'race_label_encoder.pkl'))
print('Race classes:', race_le.classes_)

## 2. Feature Engineering Helpers

In [ ]:
LETTERS = list('abcdefghijklmnopqrstuvwxyz')

def letter_features(name: str) -> np.ndarray:
    """26-dim letter-frequency vector (for RF / SVM baseline)."""
    name = str(name).lower()
    return np.array([name.count(ch) for ch in LETTERS])

## 3. Unified Prediction Functions

In [ ]:
def predict_gender_rf(first_name: str) -> dict:
    """Predict gender using the RF baseline model."""
    feats  = letter_features(first_name).reshape(1, -1)
    label  = rf_gender.predict(feats)[0]
    probs  = rf_gender.predict_proba(feats)[0]
    conf   = float(probs[list(rf_gender.classes_).index(label)])
    return {'gender': label, 'confidence': conf}


def predict_race_rf(last_name: str) -> dict:
    """Predict race using the RF baseline model."""
    feats     = letter_features(last_name).reshape(1, -1)
    idx       = rf_race.predict(feats)[0]
    label     = race_le.inverse_transform([idx])[0]
    probs     = rf_race.predict_proba(feats)[0]
    prob_dict = dict(zip(race_le.classes_, probs.round(4)))
    return {'race': label, 'probabilities': prob_dict}


def predict_gender_lgbm(first_name: str) -> dict:
    """Predict gender using the LightGBM + TF-IDF model."""
    feats  = lgbm_gender_vec.transform([first_name.lower()])
    label  = lgbm_gender.predict(feats)[0]
    probs  = lgbm_gender.predict_proba(feats)[0]
    classes = lgbm_gender.classes_
    conf   = float(probs[list(classes).index(label)])
    return {'gender': label, 'confidence': conf}


def predict_race_lgbm(last_name: str) -> dict:
    """Predict race using the LightGBM + TF-IDF model."""
    feats     = lgbm_race_vec.transform([last_name.lower()])
    idx       = lgbm_race.predict(feats)[0]
    label     = race_le.inverse_transform([idx])[0]
    probs     = lgbm_race.predict_proba(feats)[0]
    prob_dict = dict(zip(race_le.classes_, probs.round(4)))
    return {'race': label, 'probabilities': prob_dict}


def predict_demographics(first_name: str, last_name: str,
                         backend: str = 'lgbm') -> dict:
    """
    Combined gender + race prediction.

    Parameters
    ----------
    backend : 'lgbm' (default, improved) or 'rf' (baseline)
    """
    if backend == 'lgbm':
        g = predict_gender_lgbm(first_name)
        r = predict_race_lgbm(last_name)
    elif backend == 'rf':
        g = predict_gender_rf(first_name)
        r = predict_race_rf(last_name)
    else:
        raise ValueError(f"Unknown backend '{backend}'. Use 'lgbm' or 'rf'.")

    return {
        'first_name':        first_name,
        'last_name':         last_name,
        'backend':           backend,
        'gender':            g['gender'],
        'gender_confidence': g['confidence'],
        'race':              r['race'],
        'race_probabilities': r['probabilities'],
    }

## 4. Single-Name Examples

In [ ]:
for backend in ['lgbm', 'rf']:
    result = predict_demographics('Khondhaker', 'Momin', backend=backend)
    print(f'--- {backend.upper()} ---')
    print(f'  Gender : {result["gender"]}  (conf: {result["gender_confidence"]:.3f})')
    print(f'  Race   : {result["race"]}')
    top3 = sorted(result['race_probabilities'].items(), key=lambda x: -x[1])[:3]
    for cls, prob in top3:
        bar = '█' * int(prob * 30)
        print(f'    {cls:<16} {prob:.4f}  {bar}')
    print()

## 5. Batch Prediction

In [ ]:
sample_data = pd.DataFrame({
    'first_name': ['Mary',    'James',   'Maria',     'David',   'Ashley',
                   'Carlos',  'Jennifer', 'Wei',      'Fatima',  'Jordan'],
    'last_name':  ['Smith',   'Johnson',  'Garcia',   'Williams','Brown',
                   'Rodriguez','Kim',     'Chen',     'Hassan',  'Taylor'],
})
sample_data.head(10)

In [ ]:
def batch_predict(df: pd.DataFrame,
                  first_col: str = 'first_name',
                  last_col:  str = 'last_name',
                  backend:   str = 'lgbm') -> pd.DataFrame:
    """Apply gender and race predictions to every row.

    Parameters
    ----------
    backend : 'lgbm' (default, improved) or 'rf' (baseline)
    """
    first_names = df[first_col].values
    last_names  = df[last_col].values

    if backend == 'lgbm':
        # TF-IDF vectorise then predict
        Xg = lgbm_gender_vec.transform([n.lower() for n in first_names])
        Xr = lgbm_race_vec.transform([n.lower() for n in last_names])

        gender_labels = lgbm_gender.predict(Xg)
        gender_probs  = lgbm_gender.predict_proba(Xg).max(axis=1).round(4)
        race_idx      = lgbm_race.predict(Xr)
        race_labels   = race_le.inverse_transform(race_idx)

    elif backend == 'rf':
        Xg = np.array([letter_features(n) for n in first_names])
        Xr = np.array([letter_features(n) for n in last_names])

        gender_labels = rf_gender.predict(Xg)
        gender_probs  = rf_gender.predict_proba(Xg).max(axis=1).round(4)
        race_idx      = rf_race.predict(Xr)
        race_labels   = race_le.inverse_transform(race_idx)

    else:
        raise ValueError(f"Unknown backend '{backend}'.")

    out = df.copy()
    out['predicted_gender']  = gender_labels
    out['gender_confidence'] = gender_probs
    out['predicted_race']    = race_labels
    return out


results_lgbm = batch_predict(sample_data, backend='lgbm')
results_lgbm

## 6. Visualise Batch Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

g_counts = results_lgbm['predicted_gender'].value_counts()
axes[0].pie(g_counts, labels=g_counts.index, autopct='%1.0f%%',
            colors=['#DD8452', '#4C72B0'])
axes[0].set_title('Predicted Gender Distribution (LightGBM)')

r_counts = results_lgbm['predicted_race'].value_counts()
axes[1].bar(r_counts.index, r_counts.values, color='#55A868')
axes[1].set_title('Predicted Race Distribution (LightGBM)')
axes[1].set_xlabel('Race')
axes[1].set_ylabel('Count')
plt.xticks(rotation=20)

plt.tight_layout()
plt.show()

## 7. Confidence Threshold Filtering

In [ ]:
THRESHOLD = 0.70

high_conf = results_lgbm[results_lgbm['gender_confidence'] >= THRESHOLD]
print(f'Records above {THRESHOLD*100:.0f}% gender confidence: {len(high_conf)}/{len(results_lgbm)}')
high_conf[['first_name', 'last_name', 'predicted_gender', 'gender_confidence', 'predicted_race']]

## 8. Head-to-Head Comparison: RF vs LightGBM vs SVM

In [ ]:
results_rf = batch_predict(sample_data, backend='rf')

# SVM gender only (no SVM race model)
Xg_svm = np.array([letter_features(n) for n in sample_data['first_name'].values])
svm_preds = svm_gender.predict(Xg_svm)

comparison = sample_data.copy()
comparison['RF_gender']     = results_rf['predicted_gender']
comparison['LGBM_gender']   = results_lgbm['predicted_gender']
comparison['SVM_gender']    = svm_preds
comparison['RF_race']       = results_rf['predicted_race']
comparison['LGBM_race']     = results_lgbm['predicted_race']
comparison['gender_agree']  = (
    (comparison['RF_gender'] == comparison['LGBM_gender']) &
    (comparison['RF_gender'] == comparison['SVM_gender'])
)

print('All 3 gender models agree:', comparison['gender_agree'].sum(), '/', len(comparison))
print('RF vs LGBM race agree    :',
      (comparison['RF_race'] == comparison['LGBM_race']).sum(), '/', len(comparison))
print()
comparison

In [ ]:
# Confidence comparison scatter
fig, ax = plt.subplots(figsize=(6, 5))

rf_conf   = results_rf['gender_confidence'].values
lgbm_conf = results_lgbm['gender_confidence'].values
names_lab = [f'{fn} {ln}' for fn, ln in zip(sample_data['first_name'], sample_data['last_name'])]

ax.scatter(rf_conf, lgbm_conf, s=60, color='#3498db', zorder=3)
for i, name in enumerate(names_lab):
    ax.annotate(name, (rf_conf[i], lgbm_conf[i]),
                textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.plot([0.5, 1], [0.5, 1], 'k--', alpha=0.3)  # diagonal
ax.set_xlabel('RF Gender Confidence')
ax.set_ylabel('LightGBM Gender Confidence')
ax.set_title('Gender Confidence: RF vs LightGBM')
ax.set_xlim(0.5, 1.02)
ax.set_ylim(0.5, 1.02)
plt.tight_layout()
plt.show()